# Stim 用 2量子ビット Pauli ノイズ確率の生成

`main.ipynb` の「# chi 行列を生成・可視化」セルで保存した `pauli_twirled_error_probabilities.csv` を読み込み、指定した平均フォノン数 `n_bar` から `{I, X, Y, Z}^{\otimes 2}` の16個の Pauli error 確率を取得します。

CSVがまだ無い場合だけ、サンプルCSVで動作確認します。実データを見るときは、先に `main.ipynb` の該当セルを実行してください。

In [15]:
import importlib
import numpy as np
import pandas as pd

import stim_pauli_noise as spn

spn = importlib.reload(spn)
print("stim_pauli_noise:", spn.__file__)
print("pauli CSV loader ready:", hasattr(spn, "pauli_noise_from_pauli_error_csv"))


stim_pauli_noise: /Users/hirainoa/Desktop/量子/project/Moon_shot__高橋PJ/論文/stim_pauli_noise.py
pauli CSV loader ready: True


## 1. 変換したい `n_bar` を指定

In [16]:
target_n_bar = 2.0

# exact: 指定した n_bar が error_result 内にある必要があります。
# nearest: 最も近い n_bar を使います。
# interpolate: n_bar 間を線形補間します。
selection = "exact"

## 2. Pauli error CSVの保存先を指定

In [17]:
pauli_error_csv = "pauli_twirled_error_probabilities.csv"

## 3. CSVを確認

`main.ipynb` の2つ目のセルで保存されたCSVを使う。

In [18]:
from pathlib import Path

pauli_error_csv_path = Path(pauli_error_csv)
print("Pauli error CSV:", pauli_error_csv_path)
print("exists:", pauli_error_csv_path.exists())

Pauli error CSV: pauli_twirled_error_probabilities.csv
exists: True


## 5. 保存済みCSVの中身を確認

In [19]:
pauli_error_df = pd.read_csv(pauli_error_csv_path)
available_n_bars = sorted(pauli_error_df["n_bar"].unique())
print("available n_bar values:", available_n_bars)
print("rows:", len(pauli_error_df))
pauli_error_df.head(20)

available n_bar values: [0.01, 1.0, 2.0, 3.0, 4.0]
rows: 80


,n_bar,pauli,probability
0,0.01,II,9.975668e-01
1,0.01,IX,5.489618e-04
2,0.01,IY,7.138085e-05
3,0.01,IZ,1.742157e-04
4,0.01,XI,5.489645e-04
5,0.01,XX,3.526355e-04
6,0.01,XY,1.742194e-04
7,0.01,XZ,7.136753e-05
8,0.01,YI,7.138377e-05
9,0.01,YX,1.742187e-04


## 6. 指定した `n_bar` の16個の Pauli error 確率を取得

In [20]:
noise_model = spn.pauli_noise_from_pauli_error_csv(
    pauli_error_csv_path,
    n_bar=target_n_bar,
    selection=selection,
)

pauli_noise_df = pd.DataFrame(
    {
        "pauli": noise_model["pauli_labels"],
        "probability": noise_model["probabilities"],
    }
)

print("n_bar used:", noise_model["n_bar"])
print("sum probability:", pauli_noise_df["probability"].sum())
pauli_noise_df

n_bar used: 2.0
sum probability: 1.0


,pauli,probability
0,II,9.925240e-01
1,IX,1.555630e-03
2,IY,1.019610e-04
3,IZ,1.436418e-04
4,XI,1.555626e-03
5,XX,3.382090e-03
6,XY,1.436404e-04
7,XZ,1.019404e-04
8,YI,1.019566e-04
9,YX,1.436406e-04


## 7. Stim `PAULI_CHANNEL_2` 用の15確率を取得

Stim の `PAULI_CHANNEL_2` は identity `II` を含まないらしいので、`IX` から `ZZ` までの15個を渡す。

In [21]:
stim_probabilities = noise_model["stim_pauli_channel_2_probabilities"]
stim_instruction = spn.stim_pauli_channel_2_text(
    noise_model["probabilities"],
    targets=(0, 1),
)

print("number of Stim probabilities:", len(stim_probabilities))
print(stim_instruction)

number of Stim probabilities: 15
PAULI_CHANNEL_2(0.0015556300909854066, 0.00010196103328243071, 0.00014364175137493443, 0.0015556261553255999, 0.003382089696423644, 0.00014364035403040268, 0.00010194037254331203, 0.00010195658672709811, 0.00014364064368203272, 6.8314078626591237e-08, 5.1981323829057864e-08, 0.00014364097095195819, 0.00010194530987048784, 5.7429546509475825e-08, 6.377359212939028e-08) 0 1


## 8. Pauli確率テーブルもCSVとして保存

In [ ]:
output_csv = f"stim_pauli_noise_nbar_{noise_model['n_bar']}.csv"
pauli_noise_df.to_csv(output_csv, index=False)
print("Saved:", output_csv)